In [ ]:

from logging import error
import pandas as pd
import matplotlib.pyplot as plt
from chardet import detect

def get_encoding_type(file):
    with open(file, 'rb') as f:
        rawdata = f.read()
    return detect(rawdata)['encoding']

def normalize_column_and_fields(df: pd.DataFrame, column: str, new_column: str = 'municipality'):
    if column not in df.columns:
        print(f"Skipping normalization: no column {column!r}")
        return df

    df = df.copy()
    df.rename(columns={column: new_column}, inplace=True)

    df[new_column] = (
        df[new_column]
        .astype(str)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.title()
    )

    return df

In [4]:

mun_file = './Kunta.csv'
mun = csv = pd.read_csv(mun_file, encoding=get_encoding_type(mun_file))

mun.head(n=5)

,id,gml_id,natcode,namefin,nameswe,landarea,freshwarea,seawarea,totalarea
0,1,1601100116,934,Vimpeli,Vimpeli,287.24,41.46,NaN,328.70
1,2,1601100239,577,Paimio,Pemar,238.68,1.52,2.07,242.27
2,3,1601100584,402,Lapinlahti,Lapinlahti,1096.91,148.24,NaN,1245.15
3,4,1601100001,72,Hailuoto,Karlö,205.61,1.69,875.40,1082.70
4,5,1601100376,226,Karstula,Karstula,887.08,76.11,NaN,963.19


In [6]:

bed_file = './001_117s_2026_20260328-104522.csv'
bed = csv = pd.read_csv(bed_file, encoding=get_encoding_type(bed_file))

bed.head(5)

,Municipality,Type of accommodation establishment,Number of bed-places 2026*
0,WHOLE FINLAND,All accommodation establishments,190878
1,Akaa,All accommodation establishments,.
2,Alajärvi,All accommodation establishments,.
3,Alavieska,All accommodation establishments,.
4,Alavus,All accommodation establishments,364


In [8]:

road_file = './001_12jz_2024_20260328-105719.csv'
road = csv = pd.read_csv(road_file, encoding=get_encoding_type(road_file))

road.head(5)





,Alue,Tieluokka,Päällyste,Vuosi,Tiepituus
0,MA1 MANNER-SUOMI,Yhteensä,Yhteensä,2024,77827
1,MK01 Uusimaa,Yhteensä,Yhteensä,2024,4547
2,Espoo,Yhteensä,Yhteensä,2024,156
3,Helsinki,Yhteensä,Yhteensä,2024,76
4,Hyvinkää,Yhteensä,Yhteensä,2024,167


## Normalization
Now that we have all the needed data loaded, it would be good to normalize.

In [10]:

mun = normalize_column_and_fields(mun, 'namefin')
bed = normalize_column_and_fields(bed, 'Municipality')
road = normalize_column_and_fields(road, 'Alue')

print(mun.columns)
print(bed.columns)
print(road.columns)

Skipping normalization: no column 'namefin'
Skipping normalization: no column 'Municipality'
Skipping normalization: no column 'Alue'
Index(['id', 'gml_id', 'natcode', 'municipality', 'nameswe', 'landarea',
       'freshwarea', 'seawarea', 'totalarea'],
      dtype='str')
Index(['municipality', 'Type of accommodation establishment',
       'Number of bed-places 2026*'],
      dtype='str')
Index(['municipality', 'Tieluokka', 'Päällyste', 'Vuosi', 'Tiepituus'], dtype='str')


## Merge
Finally merge the dataframes into one.

In [11]:

df = pd.merge(mun, bed, on='municipality', how='left')
df = pd.merge(df, road, on='municipality', how='left')

df.head(5)

,id,gml_id,natcode,municipality,nameswe,landarea,freshwarea,seawarea,totalarea,Type of accommodation establishment,Number of bed-places 2026*,Tieluokka,Päällyste,Vuosi,Tiepituus
0,1,1601100116,934,Vimpeli,Vimpeli,287.24,41.46,NaN,328.70,All accommodation establishments,.,Yhteensä,Yhteensä,2024.0,111.0
1,2,1601100239,577,Paimio,Pemar,238.68,1.52,2.07,242.27,All accommodation establishments,.,Yhteensä,Yhteensä,2024.0,128.0
2,3,1601100584,402,Lapinlahti,Lapinlahti,1096.91,148.24,NaN,1245.15,All accommodation establishments,.,Yhteensä,Yhteensä,2024.0,468.0
3,4,1601100001,72,Hailuoto,Karlö,205.61,1.69,875.40,1082.70,All accommodation establishments,.,Yhteensä,Yhteensä,2024.0,41.0
4,5,1601100376,226,Karstula,Karstula,887.08,76.11,NaN,963.19,All accommodation establishments,.,Yhteensä,Yhteensä,2024.0,259.0
